In [ ]:
from gitsource import GithubRepositoryDataReader
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)


index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
index.fit(documents)

In [10]:
# Q1 - How many lesson pages are in the dataset?
print(len(documents))

72


In [22]:

doc = index.search("How does the agentic loop keep calling the model until it stops?")


print(f"Q2 answer: {doc[0]['filename']}")

Q2 answer: 01-agentic-rag/lessons/14-agentic-loop.md


In [25]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [50]:
INSTRUCTIONS = """
You are an assistant that can answer questions about the LLM Zoomcamp course.

As an input you have a question from a user and the course materials. You should answer the question based on the course materials.

When you don't know the answer, say "I don't know" instead of making up an answer.
"""

USER_PROMPT_TEMPLATE = """
# Question
{question}

# Context
{context}
"""

class RAGBase:
    def __init__(self, index, openai_client, model="Qwen/Qwen3.5-122B-A10B-FP8", instructions=INSTRUCTIONS):
        self.index = index
        self.openai_client = openai_client
        self.model = model
        self.instructions = instructions

    def search(self, query):
        return self.index.search(query)

    def build_user_prompt(self, docs, question):
        prompt_context = []
        for doc in docs:
            prompt_context.append("## Filename")
            prompt_context.append(doc["filename"])
            prompt_context.append("## Content")
            prompt_context.append(doc["content"])
            prompt_context.append("")

        user_prompt = USER_PROMPT_TEMPLATE.format(
            question=question,
            context="\n".join(prompt_context).strip()
        )

        return user_prompt

    def llm(self, user_prompt):

        messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": user_prompt}
        ]

        response = self.openai_client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return response

    def rag(self, question):
        # search relevant docs
        docs = self.search(question)
        # build prompt
        user_prompt = self.build_user_prompt(docs, question)
        # call llm
        return self.llm(user_prompt)






super_rag = RAGBase(index, openai_client, os.getenv("OPENAI_MODEL"))

response = super_rag.rag("How does the agentic loop keep calling the model until it stops?")



In [44]:
print("Q3. How many input (prompt) tokens did we send to the model for this request?")

print(f"PROMPT TOKENS (input): {response.usage.prompt_tokens}")
print(f"COMPLETION TOKENS (output): {response.usage.completion_tokens}")

Q3. How many input (prompt) tokens did we send to the model for this request?
PROMPT TOKENS (input): 11031
COMPLETION TOKENS (output): 674


In [40]:
from gitsource import chunk_documents

chunked_documents = chunk_documents(documents, size=2000, step=1000)

print(f"Q4. How many chunks do you get? \n {len(chunked_documents)} chunks")

Q4. How many chunks do you get? 
 295 chunks


In [42]:
chunked_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"],
)
chunked_index.fit(chunked_documents)

chunked_rag = RAGBase(chunked_index, openai_client, os.getenv("OPENAI_MODEL"))

chunked_response = chunked_rag.rag("How does the agentic loop keep calling the model until it stops?")

print("Q5. Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?")

print(f"PROMPT TOKENS (input): {chunked_response.usage.prompt_tokens}")
print(f"COMPLETION TOKENS (output): {chunked_response.usage.completion_tokens}")

Q5. Compare the input tokens with Q3. How many fewer input tokens does the chunked version send?
PROMPT TOKENS (input): 5143
COMPLETION TOKENS (output): 672


In [53]:

tools = [
    {
        "type": "function",
        "function": {
            "name": "search",
            "description": "Search the course content for entries matching the given query.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to find relevant course content.",
                    },
                },
                "required": ["query"],
                "additionalProperties": False,
            },
            "strict": True,
        },
    },
]

import json


class AgenticRAG(RAGBase):
    def llm(self, user_prompt):
    
        messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": user_prompt}
        ]

        it= 1
        while True:
            print(f"\n\nIteration {it}:\n")

            response = self.openai_client.chat.completions.create(
                model=self.model,
                messages=messages,
                tools = tools
            )
            messages.append(response.choices[0].message)

            has_tool_calls = False
            
            for item in response.choices:
                if item.finish_reason == "tool_calls":
                    for tool_call in item.message.tool_calls:
                        print(f"Tool call: {tool_call.function.name} with arguments: {tool_call.function.arguments}")
                        if tool_call.function.name == "search":
                            args = json.loads(tool_call.function.arguments)
                            search_results = chunked_rag.search(args["query"])
                    
                            # 4. Provide function call results to the model
                            messages.append(
                                {
                                    "role": "tool",
                                    "tool_call_id": tool_call.id,
                                    "content": json.dumps(search_results),
                                }
                            )
                            has_tool_calls = True
                        else:
                            raise ValueError(f"Unknown tool: {tool_call.function.name}")
    
                elif item.finish_reason == "stop":
                    print(f"Message: {item.message.content}")
                    last_message = item.message.content
                    messages.append(item.message)
    
            it += 1
            if has_tool_calls is False:
                break

        
        return response


agentic_instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

agentic_rag = AgenticRAG(chunked_index, openai_client, os.getenv("OPENAI_MODEL"), agentic_instructions)

chunked_response = agentic_rag.rag("How does the agentic loop keep calling the model until it stops?")



Iteration 1:

Tool call: search with arguments: {"query": "agentic loop while True model calls stop"}


Iteration 2:

Tool call: search with arguments: {"query": "has_function_calls flag exit condition break"}


Iteration 3:

Message: 

Based on the course content, here's how the agentic loop keeps calling the model until it stops:

## The Core Mechanism

The agentic loop uses a **`while True` loop** with a **`has_function_calls` flag** to control when to stop:

```python
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True  # 

In [ ]:
print("Q6. How many times did the agent call search?")
chunked_response = agentic_rag.rag("How does the agentic loop work, and how is it different from plain RAG?")




Iteration 1:

Tool call: search with arguments: {"query": "agentic loop while True agent loop function calls"}
Tool call: search with arguments: {"query": "plain RAG fixed flow search once LLM"}
Tool call: search with arguments: {"query": "difference between RAG and agent agentic"}


Iteration 2:

Message: 

Based on the course materials, I can explain how the agentic loop works and how it differs from plain RAG.

## Plain RAG (Fixed Flow)

In plain RAG, the pipeline follows a **fixed, predetermined flow**:

```python
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)
```

**Three steps:**
1. Search the FAQ with the exact user query
2. Build a prompt with the search results
3. Send it to the LLM for an answer

**Key characteristics:**
- The **developer decides** the flow upfront
- Search runs **once** with the exact query the user typed
- The LLM is a "passenger" - it never sees bad search resu